# C1 — Kiểm tra theo ngữ cảnh
Chạy bằng kernel Colab. C1 dùng FinancialFact từ một normalization run được chọn rõ, kiểm lại snapshot và replay normalization trước khi đánh giá. Không cần GPU; không đọc lại ảnh. Thiếu kỳ/scope, fact cần review hoặc chưa có bộ quy định được duyệt sẽ cho kết quả unknown. Không có luật giả trong cấu hình mặc định.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive')
if not root.exists():
    root = Path('/content/drive/My Drive')
assert root.is_dir(), 'Drive is not mounted'


## Lấy code mới
Notebook chỉ điều khiển; code C1 nằm trong package `financial_multiagent.validation.c1`. Ô này pull code và reload các module C1 theo thứ tự.


In [ ]:
import importlib
import os
import subprocess
import sys

repo = Path('/content/Mutil_Agent_Financial_Report')
url = 'https://github.com/VuongQuocAn/Mutil_Agent_Financial_Report.git'
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', url, str(repo)], check=True)
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
importlib.invalidate_caches()
for name in ('contracts', 'accounting', 'regulatory', 'engine', 'runner'):
    importlib.reload(importlib.import_module('financial_multiagent.validation.c1.' + name))
from financial_multiagent.validation.c1.runner import C1Input, run_c1
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)


## Chọn bundle và liệt kê normalization runs
Chỉ có một run thì có thể để ID trống. Khi có nhiều run, điền đúng ID muốn kiểm; notebook không tự chọn một bản mà bạn chưa xác nhận. Chưa có run thì chạy notebook 02 trước.


In [ ]:
TICKER = 'BTS'
YEAR = '2025'
ACCESS_SCOPE = 'demo-scope'
NORMALIZATION_RUN_ID = None

bundle_dir = root / 'Financial_MultiAgent_Results' / TICKER / YEAR / 'ocr-bundle-v1'
available_runs = sorted(p.parent.name for p in (bundle_dir / 'normalization').glob('*/normalization_report.json'))
print('Normalization run IDs:')
for name in available_runs:
    print(name)


In [ ]:
if NORMALIZATION_RUN_ID is None:
    assert len(available_runs) == 1, 'Set NORMALIZATION_RUN_ID explicitly from the list above (or run notebook 02 first)'
    selected_run_id = available_runs[0]
else:
    assert NORMALIZATION_RUN_ID in available_runs, 'Selected run does not exist'
    selected_run_id = NORMALIZATION_RUN_ID
normalization_dir = bundle_dir / 'normalization' / selected_run_id
inputs = [C1Input(bundle_dir, normalization_dir)]
print('Selected:', selected_run_id)


## Quy định và ngữ cảnh bổ sung — tùy chọn
Giữ `None` khi chưa có registry pháp lý được duyệt. C1 vẫn chạy structural/accounting/cross-document coverage, phần pháp lý ghi unknown. Registry phải có rule version, hiệu lực/ngoại lệ, predicate có kiểu và text điều khoản được lưu kèm hash. Không dùng fixture luật giả trong production. Xem schemas và tài liệu C1 để chuẩn bị registry trên Drive. Context có thể cung cấp entity attributes hoặc disclosure observations đã review; không dùng để sửa metadata FinancialFact thiếu.


In [ ]:
REGISTRY_PATH = None  # e.g. root / 'Financial_MultiAgent_Results/rules/registry.json'
CONTEXT_PATH = None   # optional scoped entity/disclosure context

# To compare another document, explicitly append its bundle and normalization run:
# inputs.append(C1Input(other_bundle_dir, other_normalization_dir))


## Chạy C1
Mỗi run ghi report, results, registry snapshot và context snapshot vào `c1/<run-id>/`. Sai scope, snapshot cũ hoặc facts không khớp replay sẽ dừng trước publish; không thay bằng pass.


In [ ]:
report = run_c1(
    inputs, access_scope=ACCESS_SCOPE,
    registry_path=REGISTRY_PATH, context_path=CONTEXT_PATH,
)
run_dir = Path(report['output_dir'])
print('Execution:', report['execution_status'])
print('Quality:', report['quality_status'])
print('Legal status:', report['legal_status'])
print('Outcomes:', report['outcome_counts'])
print('Input facts:', report['input_fact_count'])
print('Output:', run_dir)


## Xem kết quả và lý do
Accounting warning là sai lệch cần điều tra, chưa phải vi phạm pháp luật. Unknown là chưa đủ đầu vào/căn cứ để đánh giá. Nếu BTS vẫn thiếu ngày đầu kỳ/scope, các phép kiểm phụ thuộc vào chúng sẽ unknown. C1 không tự nâng document ready và không thay thế C4 ở Online.


In [ ]:
import json
from collections import Counter

results = [json.loads(line) for line in (run_dir / 'validation_results.jsonl').read_text(encoding='utf-8').splitlines()]
print('Reasons:', dict(Counter(r['reason'] for r in results)))
for result in results[:30]:
    print(result['category'], result['name'], result['outcome'], result['reason'])
    if result['details'].get('missing_metrics'):
        print('  missing:', result['details']['missing_metrics'])

for result in results:
    diagnostic = result['details'].get('numeric_diagnostic')
    if diagnostic:
        print('DIAGNOSTIC ONLY:', result['name'], result['outcome'], diagnostic)


## Xem provenance một phép kiểm
Ô dưới chỉ in refs; không tải ảnh về để hiển thị. Financial refs mở qua resolver của adapter tương ứng bundle; legal refs tra trong registry snapshot đã lưu. Ảnh/PDF gốc chưa được đọc lại để xác minh bytes ở bước C1.


In [ ]:
RESULT_INDEX = 0
result = results[RESULT_INDEX]
print(json.dumps(result, ensure_ascii=False, indent=2))
